In [1]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolAlign
from rdkit.Chem import ChemicalFeatures
from rdkit.RDConfig import RDDataDir
import os
import pandas as pd
#reading all ligands for Micobacterium tuberculosis
ligands = pd.read_excel('Mtb.xlsx', sheet_name=1)
#Name SMILES IC50 of all aurachin D homologues
ligands_Qloop = ligands[ligands['Binding site'] == 'Q-Loop']
ligands_Qloop_needed_columns = ligands_Qloop[['Name', 'SMILES', 'IC50 μM']]
#taking 1/3 of the data as training set for a model
training_set = ligands_Qloop_needed_columns[ligands_Qloop_needed_columns['IC50 μM'] < 0.3]

print(training_set)


            Name                                             SMILES  IC50 μM
2     Aurachin D  CC1=C(C(=O)C2=CC=CC=C2N1)C/C=C(\C)/CC/C=C(\C)/...    0.150
3   CK-3-22 (1T)  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)nc2)[nH]c3cccc...    0.140
8        MTD-403     Cc4c(c2ccc(N1CCCCC1)cc2)[nH]c3cc(F)cc(F)c3c4=O    0.270
9        CK-2-88          Cc4c(c2ccc(Cc1ccccc1)cc2)[nH]c3ccccc3c4=O    0.020
11       CK-2-63  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)cc2)[nH]c3cccc...    0.003
12        PG-203  Cc2[nH]c1ccccc1c(=O)c2c4ccc(Oc3ccc(OC(F)(F)F)c...    0.070
15          LT-9        O=c3cc(c2ccc(Cc1ccc(F)cc1)cc2)[nH]c4ccccc34    0.100
16        GN-171  CCOC(=O)c4c(c2ccc(Cc1ccc(OC(F)(F)F)cc1)cc2)[nH...    0.250
18       SL-2-25  Cc4c(c2ccc(c1ccc(OC(F)(F)F)cc1)nc2)[nH]c3ccccc...    0.290
19     WDH-1U-10  CCOC(=O)c4c(c2ccc(c1ccc(Cl)cc1)cc2)[nH]c3ccccc...    0.012
22      WDH-2G-6  CC(C)c4c(c2cnn(Cc1ccc(OC(F)(F)F)cc1)c2)[nH]c3c...    0.082


In [27]:
"""
Mtb Q-Loop Pharmacophore Pipeline v5.3 (Diagnostic Edition)
Prints detailed 3D coordinates and feature statistics.
"""

import os, glob, warnings
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS, ChemicalFeatures
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from collections import defaultdict

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CONFORMERS_DIR = "Conformers"
TEMPLATE_NAME  = "CK_2_63.sdf"
EXCEL_PATH     = "Mtb.xlsx"

RANDOM_SEED = 42
TRAIN_SIZE  = 0.7 
CLUSTERING_EPS = 1.6  
MIN_RADIUS = 1.0
MAX_RADIUS = 2.5
DISTANCE_CUTOFF = 1.5

FEATURE_FACTORY_FDEF = """
DefineFeature Donor [$([N;!H0;v3]),$([N;!H0;v4;+1]),$([O,S;H1;+0])]
  Family Donor
  Weights 1.0
EndFeature
DefineFeature Acceptor [$([N;H0;+0;v3]),$([O;H0;+0;v2]),$([S;H0;+0;v2])]
  Family Acceptor
  Weights 1.0
EndFeature
DefineFeature Aromatic [$([a,r6,r5])]
  Family Aromatic
  Weights 1.0
EndFeature
DefineFeature Hydrophobe [$([F,Cl,Br,I,S,C]),$([C;D2,D3,D4;!$(C=[O,N,S])])]
  Family Hydrophobe
  Weights 1.0
EndFeature
"""

# ─────────────────────────────────────────────────────────────────────────────
# CORE UTILS
# ─────────────────────────────────────────────────────────────────────────────

def find_file_fuzzy(molecule_name, directory):
    target = str(molecule_name).lower().strip().replace('.sdf', '')
    variants = {target, target.replace('-', '_'), target.replace('_', '-')}
    for filename in os.listdir(directory):
        f_lower = filename.lower()
        if f_lower.endswith('.sdf'):
            name_part = f_lower.replace('.sdf', '')
            if name_part in variants or target in name_part:
                return os.path.join(directory, filename)
    return None

def robust_load_ensemble(path, name=None):
    suppl = Chem.SDMolSupplier(path, removeHs=False, sanitize=False)
    mols = [m for m in suppl if m is not None]
    if not mols: return None
    base = mols[0]
    try: Chem.SanitizeMol(base)
    except:
        try: Chem.SanitizeMol(base, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE)
        except: return None
    for i in range(1, len(mols)):
        try: base.AddConformer(mols[i].GetConformer(), assignId=True)
        except: continue
    if name: base.SetProp("_Name", name)
    return base

def align_ensemble(probe_mol, template_mol):
    mcs = rdFMCS.FindMCS([template_mol, probe_mol], timeout=2)
    if mcs.numAtoms < 3: return probe_mol, 999.0
    patt = Chem.MolFromSmarts(mcs.smartsString)
    atom_map = list(zip(probe_mol.GetSubstructMatch(patt), template_mol.GetSubstructMatch(patt)))
    best_rmsd = 999.0
    for conf in probe_mol.GetConformers():
        try:
            rmsd = rdMolAlign.AlignMol(probe_mol, template_mol, prbCid=conf.GetId(), refCid=0, atomMap=atom_map)
            if rmsd < best_rmsd: best_rmsd = rmsd
        except: continue
    return probe_mol, best_rmsd

# ─────────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC LOGIC
# ─────────────────────────────────────────────────────────────────────────────

def extract_features_with_diagnostics(aligned_mols, factory):
    all_obs = defaultdict(list)
    for idx, (mol, pic50, name) in enumerate(aligned_mols):
        feats = factory.GetFeaturesForMol(mol)
        for conf in mol.GetConformers():
            cid = conf.GetId()
            for f in feats:
                all_obs[f.GetFamily()].append({'pos': np.array(f.GetPos(cid)), 'pIC50': pic50, 'idx': idx})

    model_points = []
    print(f"\n{'#'*20} PHARMACOPHORE MODEL SUMMARY {'#'*20}")
    print(f"{'ID':<4} {'Type':<12} {'X':<8} {'Y':<8} {'Z':<8} {'Radius':<8} {'Consensus'}")
    print("-" * 65)

    feat_counter = 1
    for family, obs in all_obs.items():
        if len(obs) < 2: continue
        coords = np.array([o['pos'] for o in obs])
        clusters = DBSCAN(eps=CLUSTERING_EPS, min_samples=2).fit(coords)
        
        for label in set(clusters.labels_):
            if label == -1: continue
            mask = clusters.labels_ == label
            c_weights = np.array([obs[i]['pIC50'] for i in np.where(mask)[0]])
            center = np.average(coords[mask], axis=0, weights=c_weights)
            
            dists = np.linalg.norm(coords[mask] - center, axis=1)
            radius = min(max(np.mean(dists) * 1.5, MIN_RADIUS), MAX_RADIUS)
            
            unique_mols = len(set(obs[i]['idx'] for i in np.where(mask)[0]))
            coverage = unique_mols / len(aligned_mols)
            
            if coverage >= 0.3:
                model_points.append({
                    'family': family, 'center': center, 'radius': radius, 
                    'coverage': coverage, 'importance': np.mean(c_weights)
                })
                print(f"{feat_counter:<4} {family:<12} {center[0]:>8.2f} {center[1]:>8.2f} {center[2]:>8.2f} {radius:>8.2f} {coverage:>10.0%}")
                feat_counter += 1
    print("#"*65)
    return sorted(model_points, key=lambda x: -x['importance'])

def score_and_count_matches(mol, points, factory):
    feats = factory.GetFeaturesForMol(mol)
    best_fit = 0.0
    matches_at_best = 0
    
    for conf in mol.GetConformers():
        cid = conf.GetId()
        c_feats = defaultdict(list)
        for f in feats: c_feats[f.GetFamily()].append(np.array(f.GetPos(cid)))
        
        point_scores = []
        matches = 0
        for pt in points:
            if pt['family'] in c_feats:
                d = min(np.linalg.norm(pos - pt['center']) for pos in c_feats[pt['family']])
                if d <= pt['radius']: matches += 1
                s = 1.0 if d <= pt['radius'] else np.exp(-((d - pt['radius'])**2) / (2 * (DISTANCE_CUTOFF**2)))
                point_scores.append(s)
            else: point_scores.append(0.0)
        
        fit = np.mean(point_scores) if point_scores else 0
        if fit > best_fit:
            best_fit = fit
            matches_at_best = matches
            
    return best_fit, matches_at_best

# ─────────────────────────────────────────────────────────────────────────────
# EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

def run():
    df = pd.read_excel(EXCEL_PATH, sheet_name=1)
    df = df[df['Binding site'].str.upper() == 'Q-LOOP'].copy()
    df['pIC50'] = pd.to_numeric(df['IC50 μM'].astype(str).str.replace('μM',''), errors='coerce').apply(
        lambda x: -np.log10(x*1e-6) if x > 0 else 0
    )
    df = df.dropna(subset=['pIC50'])
    
    train_df, val_df = train_test_split(df, train_size=TRAIN_SIZE, random_state=RANDOM_SEED)
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(FEATURE_FACTORY_FDEF)
    
    t_path = find_file_fuzzy(TEMPLATE_NAME, CONFORMERS_DIR)
    template = robust_load_ensemble(t_path)
    
    aligned_train = []
    for _, row in train_df.iterrows():
        path = find_file_fuzzy(row['Name'], CONFORMERS_DIR)
        if path:
            mol = robust_load_ensemble(path, row['Name'])
            if mol:
                _, rmsd = align_ensemble(mol, template)
                if rmsd < 5.0: aligned_train.append((mol, row['pIC50'], row['Name']))

    # Generate model with full coordinate printout
    model = extract_features_with_diagnostics(aligned_train, factory)
    
    if not model: return

    # Validation
    results = []
    for _, row in val_df.iterrows():
        path = find_file_fuzzy(row['Name'], CONFORMERS_DIR)
        if path:
            mol = robust_load_ensemble(path, row['Name'])
            if mol:
                fit, hits = score_and_count_matches(mol, model, factory)
                results.append({'name': row['Name'], 'ic50': 10**(-row['pIC50']+6), 'fit': fit, 'hits': hits})

    print(f"\n{'Name':<18} {'IC50 (μM)':<12} {'Fit Score':<10} {'Matches (X/7)'}")
    print("-" * 55)
    for r in sorted(results, key=lambda x: x['ic50']):
        print(f"{r['name']:<18} {r['ic50']:<12.4f} {r['fit']:<10.2f} {r['hits']}/{len(model)}")

if __name__ == "__main__":
    run()


#################### PHARMACOPHORE MODEL SUMMARY ####################
ID   Type         X        Y        Z        Radius   Consensus
-----------------------------------------------------------------
1    Donor            3.95     0.44     0.14     1.83       100%
2    Acceptor         5.12    -2.58     0.49     2.22        92%
3    Acceptor        -7.03    -1.00     0.64     1.82        38%
4    Acceptor        -2.96     1.86    -0.77     1.47        62%
5    Acceptor         2.94     1.81    -0.03     2.25        31%
6    Aromatic         1.59     0.55    -0.15     2.50       100%
7    Hydrophobe       0.69     0.27    -0.22     2.50       100%
#################################################################

Name               IC50 (μM)    Fit Score  Matches (X/7)
-------------------------------------------------------
WDH-2G-6           0.0820       0.72       2/7
LT-9               0.1000       0.61       3/7
RKA-310            1.4000       0.84       3/7
CK-3-23            3.60